
# Experiment 19 — Strong PatchTST + Frozen Historical Memory on Weather and Electricity

## 핵심 질문

Experiment 18에서 Weather와 Electricity의 강한 official-level PatchTST baseline을 확정했습니다.

이제 그 checkpoint를 그대로 고정한 상태에서 기존 historical-memory module이 추가적인 예측 이득을 제공하는지 검증합니다.

$$
\boxed{
\text{Strong PatchTST}
+
\text{Frozen Historical Memory}
>
\text{Strong PatchTST}
}
$$

대상은 다음과 같습니다.

$$
H \in \{96, 192, 336, 720\}
$$

- Weather
- Electricity

---

## 이번 실험에서 변경하지 않는 요소

Retrieval 쪽은 Experiment 13에서 확정한 설정을 그대로 사용합니다.

### Predictive retriever

$$
s(q,i)
=
\gamma \cos(z_q,z_i)
$$

- EmbeddingOnly predictive retriever
- retrieval query length: \(L_R=96\)
- patch length: 16
- patch stride: 16
- embedding dimension: 64
- Top-\(K=10\)
- retrieval-memory stride: 24
- retrieved future: uniform Top-10 average

### Strong direct forecaster

Experiment 18의 official PatchTST checkpoint를 그대로 불러옵니다.

$$
L_D=336
$$

PatchTST는 이번 notebook에서 full model을 다시 학습하지 않습니다.

### Adaptive trust

Experiment 16/17과 같은 26-dimensional gate를 사용합니다.

$$
\hat{\mathbf y}
=
(1-\alpha_q)
\hat{\mathbf y}_{\mathrm{PatchTST}}
+
\alpha_q
\hat{\mathbf y}_{\mathrm{retrieval}}
$$

그리고 validation-calibrated shrinkage를 적용합니다.

$$
\alpha_{\mathrm{final}}
=
(1-\lambda)\alpha_0
+
\lambda\alpha_{\mathrm{gate}}
$$

---

## Leakage-free cross-fitting

Gate가 strong PatchTST의 error regime을 학습하도록
train split 내부에 세 개의 chronological fold를 만듭니다.

$$
0.55 \rightarrow 0.70
$$

$$
0.70 \rightarrow 0.85
$$

$$
0.85 \rightarrow 1.00
$$

각 fold에서:

1. prefix 구간만으로 normalization
2. prefix 구간만으로 official PatchTST fold model 학습
3. prefix 이전에 future까지 완전히 관측된 history만 retrieval memory로 사용
4. 바로 뒤 OOF 구간의 prediction과 retrieval을 생성
5. OOF target은 gate 학습에만 사용

Fold direct model의 epoch 수는 OOF target으로 고르지 않습니다.

Experiment 18의 full official PatchTST가 validation으로 선택한 best epoch 수를 고정하여 사용합니다.

$$
E_{\mathrm{fold}}
=
E_{\mathrm{full}}^{*}
$$

---

## Validation과 test

### Validation

- direct model: Experiment 18 train-only checkpoint
- retrieval memory: train histories only
- scalar \(\alpha_0\): validation only
- gate epoch: validation only
- shrinkage \(\lambda\): validation only

### Test

- direct model: Experiment 18 train-only checkpoint
- retrieval memory: train + validation histories only
- test history를 rolling memory에 추가하지 않음
- all valid test origins
- stride 1
- all channels

---

## 계산량 절약

Electricity는 321 channels이므로 cross-fitting이 매우 무겁습니다.

이 notebook은 다음을 모두 cache합니다.

- fold official PatchTST checkpoint
- fold OOF gate features
- validation gate features
- retrieval-memory embeddings
- gate checkpoint
- final per-anchor test losses
- condition별 summary

중간에 kernel이 종료되어도 `RESUME=True`로 이어서 실행할 수 있습니다.

완료된 dataset × horizon 조건은 자동으로 건너뜁니다.


In [1]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 460)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASETS = ["Weather", "Electricity"]
HORIZONS = [96, 192, 336, 720]

# Weather first: cheaper and gives an early scientific signal.
TASKS = [
    ("Weather", 96),
    ("Weather", 192),
    ("Weather", 336),
    ("Weather", 720),
    ("Electricity", 96),
    ("Electricity", 192),
    ("Electricity", 336),
    ("Electricity", 720),
]

DIRECT_SEQ_LEN = 336
RET_SEQ_LEN = 96

CROSSFIT_SEED = 1313

TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DIM = 64
REP_DROPOUT = 0.1

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Frozen adaptive gate.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

EPS = 1e-8
RETRIEVER_USE_AMP = torch.cuda.is_available()

# Keep roughly the same number of channel-query pairs per batch.
TARGET_QUERY_PAIRS = 672

EXP18_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_baselines"
)

FROZEN_RET_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening"
)

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_plus_frozen_retrieval"
)

DIRS = {
    "fold_direct": ROOT / "fold_direct",
    "oof": ROOT / "oof",
    "validation": ROOT / "validation",
    "memory_emb": ROOT / "memory_embeddings",
    "gate": ROOT / "gate",
    "history": ROOT / "history",
    "paired": ROOT / "paired_test",
    "channel": ROOT / "channel_test",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE = False

print("Device:", DEVICE)
print("Tasks:", TASKS)
print("Experiment 18:", EXP18_ROOT)
print("Frozen retrievers:", FROZEN_RET_ROOT)
print("Output:", ROOT)


Device: cuda
Tasks: [('Weather', 96), ('Weather', 192), ('Weather', 336), ('Weather', 720), ('Electricity', 96), ('Electricity', 192), ('Electricity', 336), ('Electricity', 720)]
Experiment 18: /data/dataset/strong_forecaster/weather_electricity_official_patchtst_baselines
Frozen retrievers: /data/dataset/strong_forecaster/multidataset_crossfit_screening
Output: /data/dataset/strong_forecaster/weather_electricity_official_patchtst_plus_frozen_retrieval


## 1. Load the same official PatchTST implementation used in Experiment 18

In [2]:

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p
        for p in OFFICIAL_REPO_CANDIDATES
        if (
            p
            / "PatchTST_supervised"
            / "models"
            / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    raise FileNotFoundError(
        "Official PatchTST repository was not found. "
        "Experiment 18 should have used the same repository."
    )

SUPERVISED_ROOT = (
    OFFICIAL_REPO
    / "PatchTST_supervised"
)

for module_name in list(
    sys.modules.keys()
):
    if (
        module_name == "models"
        or module_name.startswith("models.")
        or module_name == "layers"
        or module_name.startswith("layers.")
    ):
        del sys.modules[module_name]

if str(SUPERVISED_ROOT) in sys.path:
    sys.path.remove(
        str(SUPERVISED_ROOT)
    )

sys.path.insert(
    0,
    str(SUPERVISED_ROOT),
)

patchtst_module = importlib.import_module(
    "models.PatchTST"
)

OfficialPatchTST = patchtst_module.Model

actual_model_file = Path(
    patchtst_module.__file__
).resolve()

expected_model_file = (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).resolve()

print("Imported:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong PatchTST implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official PatchTST implementation is active."
)


Imported: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py
PASS: official PatchTST implementation is active.


## 2. Load Experiment 18 dataset paths and reproduce its exact data ordering

In [3]:

EXP18_SUMMARY_PATH = (
    EXP18_ROOT
    / "summary.csv"
)

if not EXP18_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Experiment 18 summary not found: {EXP18_SUMMARY_PATH}"
    )

EXP18_SUMMARY = pd.read_csv(
    EXP18_SUMMARY_PATH
)

display(
    EXP18_SUMMARY[
        [
            "Dataset",
            "Horizon",
            "FullStride1_MSE",
            "FullStride1_MAE",
            "BestEpoch",
        ]
    ].sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
)


DATA_PATH_CANDIDATES = {
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather.csv"
        ),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path(
            "/data/dataset/electricity/"
            "electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity.csv"
        ),
    ],
}

DATA_PATHS = {}

for name, candidates in DATA_PATH_CANDIDATES.items():
    hit = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    DATA_PATHS[name] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

if any(
    p is None
    for p in DATA_PATHS.values()
):
    raise FileNotFoundError(
        "Weather/Electricity CSV path not found. "
        "Use the same dataset paths as Experiment 18."
    )


EXPECTED_CHANNELS = {
    "Weather": 21,
    "Electricity": 321,
}


def prepare_custom_data(
    name,
):
    path = DATA_PATHS[name]

    df_raw = pd.read_csv(
        path
    )

    if "date" not in df_raw.columns:
        raise ValueError(
            f"{name}: expected a date column."
        )

    if "OT" not in df_raw.columns:
        raise ValueError(
            f"{name}: expected OT target column."
        )

    # Exact Experiment 18 / official Dataset_Custom ordering:
    # date, all other variables, OT.
    cols = list(
        df_raw.columns
    )

    cols.remove(
        "OT"
    )

    cols.remove(
        "date"
    )

    ordered = df_raw[
        ["date"]
        + cols
        + ["OT"]
    ].copy()

    value_cols = list(
        ordered.columns[
            1:
        ]
    )

    raw = ordered[
        value_cols
    ].to_numpy(
        dtype=np.float32
    )

    n = len(
        raw
    )

    num_train = int(
        n
        * 0.7
    )

    num_test = int(
        n
        * 0.2
    )

    num_val = (
        n
        - num_train
        - num_test
    )

    train_end = num_train
    val_end = (
        num_train
        + num_val
    )

    test_end = n

    mu = raw[
        :train_end
    ].mean(
        axis=0
    ).astype(
        np.float32
    )

    sd = raw[
        :train_end
    ].std(
        axis=0
    ).astype(
        np.float32
    )

    if np.any(
        sd <= 1e-6
    ):
        raise ValueError(
            f"{name}: degenerate train channel."
        )

    z = (
        (
            raw
            - mu[
                None,
                :
            ]
        )
        / sd[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    if (
        raw.shape[
            1
        ]
        != EXPECTED_CHANNELS[
            name
        ]
    ):
        raise ValueError(
            f"{name}: expected "
            f"{EXPECTED_CHANNELS[name]} channels, "
            f"found {raw.shape[1]}."
        )

    return {
        "name": name,
        "path": path,
        "columns": value_cols,
        "raw": raw,
        "z": z,
        "mean": mu,
        "std": sd,
        "n_channels": raw.shape[
            1
        ],
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
        "num_train": num_train,
        "num_val": num_val,
        "num_test": num_test,
    }


DATA = {
    name:
        prepare_custom_data(
            name
        )
    for name in DATASETS
}

display(
    pd.DataFrame([
        {
            "Dataset": name,
            "Path": str(
                d[
                    "path"
                ]
            ),
            "Rows": len(
                d[
                    "raw"
                ]
            ),
            "Channels": d[
                "n_channels"
            ],
            "TrainEnd": d[
                "train_end"
            ],
            "ValEnd": d[
                "val_end"
            ],
            "TestEnd": d[
                "test_end"
            ],
        }
        for name, d
        in DATA.items()
    ])
)


,Dataset,Horizon,FullStride1_MSE,FullStride1_MAE,BestEpoch
4,Electricity,96,0.129977,0.222527,97
5,Electricity,192,0.149116,0.242038,48
6,Electricity,336,0.165690,0.258694,69
7,Electricity,720,0.203216,0.292492,39
0,Weather,96,0.149690,0.197816,65
1,Weather,192,0.194775,0.240853,45
2,Weather,336,0.246900,0.281548,19
3,Weather,720,0.320894,0.334351,23


Weather    : /data/Time-Series-Library/dataset/weather/weather.csv
Electricity: /data/Time-Series-Library/dataset/electricity/electricity.csv


,Dataset,Path,Rows,Channels,TrainEnd,ValEnd,TestEnd
0,Weather,/data/Time-Series-Library/dataset/weather/weat...,52696,21,36887,42157,52696
1,Electricity,/data/Time-Series-Library/dataset/electricity/...,26304,321,18412,21044,26304


## 3. Official PatchTST recipes and Experiment 18 checkpoint loader

In [4]:

RECIPES = {
    "Weather": {
        "enc_in": 21,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "learning_rate": 1e-4,
        "lradj": "type3",
        "pct_start": 0.3,
    },
    "Electricity": {
        "enc_in": 321,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 32,
        "train_epochs": 100,
        "learning_rate": 1e-4,
        "lradj": "TST",
        "pct_start": 0.2,
    },
}


def official_config(
    name,
    horizon,
):
    r = RECIPES[
        name
    ]

    return SimpleNamespace(
        enc_in=
            r[
                "enc_in"
            ],
        seq_len=
            DIRECT_SEQ_LEN,
        pred_len=
            int(
                horizon
            ),
        e_layers=
            r[
                "e_layers"
            ],
        n_heads=
            r[
                "n_heads"
            ],
        d_model=
            r[
                "d_model"
            ],
        d_ff=
            r[
                "d_ff"
            ],
        dropout=
            r[
                "dropout"
            ],
        fc_dropout=
            r[
                "fc_dropout"
            ],
        head_dropout=
            r[
                "head_dropout"
            ],
        individual=0,
        patch_len=
            r[
                "patch_len"
            ],
        stride=
            r[
                "stride"
            ],
        padding_patch=
            "end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_official_model(
    name,
    horizon,
):
    return OfficialPatchTST(
        official_config(
            name,
            horizon,
        )
    ).float().to(
        DEVICE
    )


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def exp18_direct_ckpt_path(
    name,
    horizon,
):
    return (
        EXP18_ROOT
        / "full_direct"
        / (
            f"{name.lower()}_L336_H{horizon}_"
            "official_recipe_seed2021.pt"
        )
    )


def load_exp18_direct(
    name,
    horizon,
):
    path = exp18_direct_ckpt_path(
        name,
        horizon,
    )

    ckpt = load_torch(
        path
    )

    model = build_official_model(
        name,
        horizon,
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


def exp18_reference(
    name,
    horizon,
):
    hit = EXP18_SUMMARY[
        (
            EXP18_SUMMARY[
                "Dataset"
            ]
            == name
        )
        & (
            EXP18_SUMMARY[
                "Horizon"
            ]
            == horizon
        )
    ]

    if len(
        hit
    ) != 1:
        raise RuntimeError(
            f"Expected exactly one Experiment 18 row "
            f"for {name} H={horizon}."
        )

    return hit.iloc[
        0
    ]


## 4. Frozen retriever and gate definitions

In [5]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def full_retriever_ckpt_path(
    name,
    horizon,
):
    return (
        FROZEN_RET_ROOT
        / "full_retriever"
        / (
            f"{name}_H{horizon}_seed0.pt"
        )
    )


def fold_retriever_ckpt_path(
    name,
    horizon,
    fold,
):
    return (
        FROZEN_RET_ROOT
        / "fold_retriever"
        / (
            f"{name}_H{horizon}_F{fold}_seed0.pt"
        )
    )


def load_frozen_retriever(
    path,
):
    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


## 5. Preflight — stop before expensive computation if anything is missing

In [6]:

preflight_rows = []

for name, horizon in TASKS:
    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Experiment18 direct",
        "Path":
            exp18_direct_ckpt_path(
                name,
                horizon,
            ),
    })

    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Full frozen retriever",
        "Path":
            full_retriever_ckpt_path(
                name,
                horizon,
            ),
    })

    for fold in range(
        1,
        4,
    ):
        preflight_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Kind":
                f"Fold retriever F{fold}",
            "Path":
                fold_retriever_ckpt_path(
                    name,
                    horizon,
                    fold,
                ),
        })

preflight = pd.DataFrame(
    preflight_rows
)

preflight[
    "Exists"
] = preflight[
    "Path"
].map(
    lambda p:
        Path(
            p
        ).is_file()
)

display(
    preflight
)

missing = preflight[
    ~preflight[
        "Exists"
    ]
]

if len(
    missing
):
    print(
        "\nMissing prerequisites:"
    )

    for p in missing[
        "Path"
    ]:
        print(
            " -",
            p,
        )

    raise FileNotFoundError(
        "Required Experiment 18 or frozen Experiment 13 checkpoint "
        "is missing. Do not silently retrain a different retriever."
    )

print(
    "PASS: every prerequisite checkpoint is present."
)


,Dataset,Horizon,Kind,Path,Exists
0,Weather,96,Experiment18 direct,/data/dataset/strong_forecaster/weather_electr...,True
1,Weather,96,Full frozen retriever,/data/dataset/strong_forecaster/multidataset_c...,True
2,Weather,96,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
3,Weather,96,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True
4,Weather,96,Fold retriever F3,/data/dataset/strong_forecaster/multidataset_c...,True
5,Weather,192,Experiment18 direct,/data/dataset/strong_forecaster/weather_electr...,True
6,Weather,192,Full frozen retriever,/data/dataset/strong_forecaster/multidataset_c...,True
7,Weather,192,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
8,Weather,192,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True
9,Weather,192,Fold retriever F3,/data/dataset/strong_forecaster/multidataset_c...,True


PASS: every prerequisite checkpoint is present.


## 6. Core time-series utilities

In [7]:

def query_anchor_batch(
    name,
):
    c = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_QUERY_PAIRS
        // c,
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
):
    return np.arange(
        max(
            int(
                start
            ),
            DIRECT_SEQ_LEN,
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.benchmark = False


def prefix_norm(
    raw,
    prefix,
):
    mu = raw[
        :prefix
    ].mean(
        axis=0
    ).astype(
        np.float32
    )

    sd = raw[
        :prefix
    ].std(
        axis=0
    ).astype(
        np.float32
    )

    if np.any(
        sd <= 1e-6
    ):
        raise ValueError(
            "Degenerate prefix channel."
        )

    return (
        (
            raw
            - mu[
                None,
                :
            ]
        )
        / sd[
            None,
            :
        ]
    ).astype(
        np.float32
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible retrieval memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


## 7. Cached retrieval-memory embeddings

In [8]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_cache_path(
    name,
    horizon,
    tag,
):
    return (
        DIRS[
            "memory_emb"
        ]
        / (
            f"{name}_H{horizon}_{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_cache_path(
        name,
        horizon,
        tag,
    )

    expected_shape = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path
        )

        if (
            tuple(
                candidate.shape
            )
            == expected_shape
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding cache:",
                path.name,
            )

    if emb_np is None:
        print(
            "Building memory embedding cache:",
            path.name,
            expected_shape,
        )

        emb_np = np.empty(
            expected_shape,
            dtype=np.float32,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or c + 1
                == channels
            ):
                print(
                    f"  encoded channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


## 8. Frozen retriever inference

In [9]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    past, pattern, ctx, true = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }



## 9. Strong PatchTST residual prediction

Retriever는 query의 마지막 관측값을 기준으로 future residual을 예측합니다.

따라서 PatchTST도 동일한 좌표로 변환합니다.

$$
\Delta
\hat{\mathbf y}_{\mathrm{direct}}
=
\hat{\mathbf y}_{\mathrm{direct}}
-
\mathbf x_{t}
$$

같은 current value를 빼므로 residual space의 MSE는 original normalized-value space의 MSE와 동일합니다.


In [10]:

@torch.no_grad()
def direct_residual(
    model,
    z,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    x = torch.from_numpy(
        z[
            pi,
            :
        ].astype(
            np.float32
        )
    ).to(
        DEVICE
    )

    pred = model(
        x
    ).float()

    residual = (
        pred
        - x[
            :,
            -1:,
            :
        ].float()
    )

    del (
        x,
        pred,
    )

    return residual


## 10. Exact official fold-direct training

In [11]:

def adjust_type3_lr(
    optimizer,
    base_lr,
    epoch,
):
    lr = (
        base_lr
        if epoch < 3
        else base_lr
        * (
            0.9
            ** (
                epoch
                - 3
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


def fold_direct_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_H{horizon}_F{fold}_"
            "official_direct.pt"
        )
    )


def direct_train_anchors(
    end,
    horizon,
):
    return np.arange(
        DIRECT_SEQ_LEN,
        int(
            end
        )
        - horizon
        + 1,
        dtype=np.int64,
    )


def make_direct_batch(
    z,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    return (
        z[
            pi,
            :
        ].astype(
            np.float32
        ),
        z[
            fi,
            :
        ].astype(
            np.float32
        ),
    )


def train_fold_official_direct(
    name,
    horizon,
    z,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        name,
        horizon,
        fold,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = build_official_model(
            name,
            horizon,
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded fold direct:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    r = RECIPES[
        name
    ]

    seed = (
        CROSSFIT_SEED
        + horizon
        * 100
        + fold
        * 13
        + int(
            prefix
        )
        % 997
    )

    set_seed(
        seed
    )

    model = build_official_model(
        name,
        horizon,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
    )

    train_aa = direct_train_anchors(
        prefix,
        horizon,
    )

    steps_per_epoch = (
        len(
            train_aa
        )
        // r[
            "batch_size"
        ]
    )

    if steps_per_epoch < 1:
        raise ValueError(
            f"{name} H={horizon} F{fold}: "
            "insufficient training windows."
        )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=
            optimizer,
        steps_per_epoch=
            steps_per_epoch,
        pct_start=
            r[
                "pct_start"
            ],
        epochs=
            r[
                "train_epochs"
            ],
        max_lr=
            r[
                "learning_rate"
            ],
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    history = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        order = rng.permutation(
            len(
                train_aa
            )
        )

        usable = (
            len(
                order
            )
            // r[
                "batch_size"
            ]
        ) * r[
            "batch_size"
        ]

        order = order[
            :usable
        ]

        losses = []

        for left in range(
            0,
            usable,
            r[
                "batch_size"
            ],
        ):
            ids = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            x_np, y_np = make_direct_batch(
                z,
                train_aa[
                    ids
                ],
                horizon,
            )

            x = torch.from_numpy(
                x_np
            ).to(
                DEVICE
            )

            y = torch.from_numpy(
                y_np
            ).to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(
                x
            )

            loss = F.mse_loss(
                pred,
                y,
            )

            loss.backward()

            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            if r[
                "lradj"
            ] == "TST":
                current = onecycle.get_last_lr()[
                    0
                ]

                for group in optimizer.param_groups:
                    group[
                        "lr"
                    ] = current

                onecycle.step()

            del (
                x,
                y,
                pred,
                loss,
            )

        if r[
            "lradj"
        ] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[
                0
            ]

        train_mse = float(
            np.mean(
                losses
            )
        )

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                train_mse,
            "LR":
                lr,
        })

        print(
            f"Fold direct {name:11s} "
            f"H={horizon:3d} F{fold} "
            f"ep={epoch:03d}/{fixed_epochs} "
            f"train={train_mse:.6f} "
            f"lr={lr:.3e}"
        )

        pd.DataFrame(
            history
        ).to_csv(
            DIRS[
                "history"
            ]
            / (
                f"{name}_H{horizon}_F{fold}_"
                "direct_history.csv"
            ),
            index=False,
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


## 11. Frozen 26-dimensional gate features

In [12]:

def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


## 12. Cached OOF and validation feature collection

In [13]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    C = data[
        "n_channels"
    ]

    batch_anchors = query_anchor_batch(
        data[
            "name"
        ]
    )

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for i in range(
        0,
        len(
            anchors
        ),
        batch_anchors,
    ):
        a = anchors[
            i:
            i+batch_anchors
        ]

        A = len(
            a
        )

        d3 = direct_residual(
            direct_model,
            z,
            a,
            horizon,
        )

        pair_anchor = np.repeat(
            a,
            C,
        )

        pair_channel = np.tile(
            np.arange(
                C,
                dtype=np.int64,
            ),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            z,
            pair_anchor,
            pair_channel,
            horizon,
        )

        retrieval = r[
            "cand"
        ].mean(
            dim=1
        )

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(
            -1,
            horizon,
        )

        true = r[
            "true"
        ]

        feat = gate_features(
            r,
            retrieval,
            direct,
        )

        abc = abc_terms(
            direct,
            retrieval,
            true,
        )

        features.append(
            feat.cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        abcs.append(
            abc.cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        anchors_out.append(
            pair_anchor
        )

        channels_out.append(
            pair_channel
        )

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            feat,
            abc,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


def oof_cache_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H{horizon}_F{fold}_"
            "official_direct.npz"
        )
    )


def val_cache_path(
    name,
    horizon,
):
    return (
        DIRS[
            "validation"
        ]
        / (
            f"{name}_H{horizon}_"
            "validation_features.npz"
        )
    )


def build_oof_fold(
    data,
    horizon,
    fold,
    p0,
    p1,
    direct_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_cache_path(
        name,
        horizon,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            "Loaded OOF cache:",
            out_path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z = prefix_norm(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = train_fold_official_direct(
        name,
        horizon,
        z,
        prefix,
        direct_epochs,
        fold,
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            horizon,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        horizon,
    )

    tag = (
        f"F{fold}_prefix{prefix}"
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        tag,
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={horizon} F{fold}: "
        f"prefix={prefix}, "
        f"end={oof_end}, "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def build_validation_cache(
    data,
    horizon,
    direct_model,
    retriever,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    path = val_cache_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            path
        )

        print(
            "Loaded validation cache:",
            path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        "validation_train_memory",
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        horizon,
        stride=1,
    )

    print(
        f"Validation {name} H={horizon}: "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        data[
            "z"
        ],
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


## 13. Cross-fitted gate training and validation-only calibration

In [14]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "official_direct.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} "
            f"H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and refit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )



## 14. Final test, Oracle diagnostic, and paired bootstrap

Oracle은 실사용 방법이 아니라 diagnostic입니다.

현재 retriever가 만든 동일한 retrieval forecast와 PatchTST forecast 사이에서
test future를 알고 최적 convex weight를 선택합니다.

따라서 Oracle headroom은

$$
\boxed{
\text{현재 retrieval forecast에 보완 정보가 존재하는가?}
}
$$

를 진단합니다.

실제 방법의 성능 주장은 Oracle이 아니라 ShrinkAdaptive로 판단합니다.


In [15]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    C = data[
        "n_channels"
    ]

    batch_anchors = query_anchor_batch(
        data[
            "name"
        ]
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    for i in range(
        0,
        len(
            anchors
        ),
        batch_anchors,
    ):
        a = anchors[
            i:
            i+batch_anchors
        ]

        A = len(
            a
        )

        d3 = direct_residual(
            direct_model,
            data[
                "z"
            ],
            a,
            horizon,
        )

        pair_anchor = np.repeat(
            a,
            C,
        )

        pair_channel = np.tile(
            np.arange(
                C,
                dtype=np.int64,
            ),
            A,
        )

        r = retrieve(
            retriever,
            memory_gpu_obj,
            data[
                "z"
            ],
            pair_anchor,
            pair_channel,
            horizon,
        )

        retrieval = r[
            "cand"
        ].mean(
            dim=1
        )

        direct = d3.permute(
            0,
            2,
            1,
        ).reshape(
            -1,
            horizon,
        )

        true = r[
            "true"
        ]

        scalar = (
            direct
            + scalar_alpha
            * (
                retrieval
                - direct
            )
        )

        features = gate_features(
            r,
            retrieval,
            direct,
        ).cpu().numpy().astype(
            np.float32
        )

        scaled = scale_features(
            features,
            gate_ckpt[
                "FeatureMedian"
            ],
            gate_ckpt[
                "FeatureIQR"
            ],
        )

        gate_alpha_values = gate(
            torch.from_numpy(
                scaled
            ).to(
                DEVICE
            )
        )

        shrink_alpha_values = (
            (
                1.0
                - shrink_lambda
            )
            * scalar_alpha
            + shrink_lambda
            * gate_alpha_values
        )

        raw_adaptive = (
            direct
            + gate_alpha_values[
                :,
                None
            ]
            * (
                retrieval
                - direct
            )
        )

        shrink_adaptive = (
            direct
            + shrink_alpha_values[
                :,
                None
            ]
            * (
                retrieval
                - direct
            )
        )

        oracle_alpha, oracle = (
            oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )
        )

        predictions = {
            "Direct":
                direct,
            "Retrieval":
                retrieval,
            "Scalar":
                scalar,
            "RawAdaptive":
                raw_adaptive,
            "ShrinkAdaptive":
                shrink_adaptive,
            "Oracle":
                oracle,
        }

        for key, pred in predictions.items():
            update_stat(
                stats[
                    key
                ],
                pred,
                true,
            )

            per_pair = (
                (
                    pred
                    - true
                )
                ** 2
            ).mean(
                dim=1
            ).reshape(
                A,
                C,
            ).mean(
                dim=1
            )

            anchor_mse[
                key
            ].extend(
                per_pair.cpu()
                .numpy()
                .tolist()
            )

        # Per-channel diagnostics.
        direct_e2 = (
            (
                direct
                - true
            )
            ** 2
        ).reshape(
            A,
            C,
            horizon,
        )

        shrink_e2 = (
            (
                shrink_adaptive
                - true
            )
            ** 2
        ).reshape(
            A,
            C,
            horizon,
        )

        channel_sse_direct += (
            direct_e2.sum(
                dim=(
                    0,
                    2,
                )
            ).cpu()
            .numpy()
        )

        channel_sse_shrink += (
            shrink_e2.sum(
                dim=(
                    0,
                    2,
                )
            ).cpu()
            .numpy()
        )

        channel_count += (
            A
            * horizon
        )

        raw_alpha_sum += float(
            gate_alpha_values.sum()
        )

        shrink_alpha_sum += float(
            shrink_alpha_values.sum()
        )

        oracle_alpha_sum += float(
            oracle_alpha.sum()
        )

        oracle_positive += int(
            (
                oracle_alpha
                > 0.01
            ).sum()
        )

        n_pairs += len(
            gate_alpha_values
        )

        del (
            d3,
            r,
            retrieval,
            direct,
            true,
            scalar,
            features,
            scaled,
            gate_alpha_values,
            shrink_alpha_values,
            raw_adaptive,
            shrink_adaptive,
            oracle_alpha,
            oracle,
            predictions,
            direct_e2,
            shrink_e2,
        )

        if (
            i == 0
            or (
                i
                // batch_anchors
                + 1
            )
            % 100
            == 0
            or (
                i
                + batch_anchors
                >= len(
                    anchors
                )
            )
        ):
            print(
                f"  test anchors "
                f"{min(i+batch_anchors, len(anchors))}"
                f"/{len(anchors)}"
            )

    channel_direct = (
        channel_sse_direct
        / channel_count
    )

    channel_shrink = (
        channel_sse_shrink
        / channel_count
    )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_direct,
        "channel_shrink_mse":
            channel_shrink,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=131313,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }



## 15. Main Experiment

각 조건에서 순서는 다음과 같습니다.

1. Experiment 18 strong PatchTST load
2. horizon-specific frozen retriever load
3. three OOF folds 생성 또는 cache load
4. validation feature 생성 또는 cache load
5. gate / scalar / shrinkage validation calibration
6. static train+validation retrieval memory로 all-window test
7. bootstrap 및 channel diagnostic 저장

조건 하나가 끝날 때마다 `summary.csv`를 즉시 갱신합니다.


In [ ]:

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

CALIBRATION_PATH = (
    ROOT
    / "calibration.csv"
)

FOLD_PATH = (
    ROOT
    / "folds.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_rows = (
    pd.read_csv(
        BOOTSTRAP_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and BOOTSTRAP_PATH.exists()
    )
    else []
)

fold_rows = (
    pd.read_csv(
        FOLD_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and FOLD_PATH.exists()
    )
    else []
)

calibration_frames = []


def already_done(
    name,
    horizon,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            (
                existing[
                    "Dataset"
                ]
                == name
            )
            & (
                existing[
                    "Horizon"
                ]
                == horizon
            )
        ).any()
    )


for name, horizon in TASKS:
    if already_done(
        name,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{name} H={horizon}"
        )
        continue

    start_time = time.time()

    data = DATA[
        name
    ]

    C = data[
        "n_channels"
    ]

    print(
        "\n"
        + "#"
        * 150
    )

    print(
        f"{name} | H={horizon} | "
        "OFFICIAL PATCHTST + FROZEN HISTORICAL MEMORY"
    )

    print(
        "#"
        * 150
    )

    # --------------------------------------------------------
    # 1. Load the exact Experiment 18 direct checkpoint.
    # --------------------------------------------------------
    direct_model, direct_ckpt = load_exp18_direct(
        name,
        horizon,
    )

    ref = exp18_reference(
        name,
        horizon,
    )

    direct_epochs = int(
        direct_ckpt[
            "BestEpoch"
        ]
    )

    print(
        f"Experiment 18 reference: "
        f"MSE={float(ref['FullStride1_MSE']):.6f}, "
        f"MAE={float(ref['FullStride1_MAE']):.6f}, "
        f"best_epoch={direct_epochs}"
    )

    # --------------------------------------------------------
    # 2. Load frozen full retriever.
    # --------------------------------------------------------
    retriever, retriever_ckpt = load_frozen_retriever(
        full_retriever_ckpt_path(
            name,
            horizon,
        )
    )

    print(
        "Frozen retriever best epoch:",
        retriever_ckpt.get(
            "BestEpoch",
            "unknown",
        ),
    )

    # --------------------------------------------------------
    # 3. Chronological OOF data.
    # --------------------------------------------------------
    oof_parts = []

    for fold, (
        p0,
        p1,
    ) in enumerate(
        FOLDS,
        start=1,
    ):
        part = build_oof_fold(
            data,
            horizon,
            fold,
            p0,
            p1,
            direct_epochs,
        )

        oof_parts.append(
            part
        )

        fold_rows = [
            r
            for r in fold_rows
            if not (
                r.get(
                    "Dataset"
                )
                == name
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
                and int(
                    r.get(
                        "Fold",
                        -1,
                    )
                )
                == fold
            )
        ]

        fold_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Fold":
                fold,
            "PrefixFrac":
                p0,
            "OOFEndFrac":
                p1,
            "Pairs":
                len(
                    part[
                        "feature"
                    ]
                ),
            "Anchors":
                len(
                    np.unique(
                        part[
                            "anchor"
                        ]
                    )
                ),
            "DirectFixedEpochs":
                direct_epochs,
            "RetrieverCheckpoint":
                str(
                    fold_retriever_ckpt_path(
                        name,
                        horizon,
                        fold,
                    )
                ),
        })

        pd.DataFrame(
            fold_rows
        ).to_csv(
            FOLD_PATH,
            index=False,
        )

    oof_x = np.concatenate(
        [
            part[
                "feature"
            ]
            for part in oof_parts
        ],
        axis=0,
    )

    oof_abc = np.concatenate(
        [
            part[
                "abc"
            ]
            for part in oof_parts
        ],
        axis=0,
    )

    print(
        "Total OOF pairs:",
        len(
            oof_x
        ),
    )

    # --------------------------------------------------------
    # 4. Validation cache and calibration.
    # --------------------------------------------------------
    val_data = build_validation_cache(
        data,
        horizon,
        direct_model,
        retriever,
    )

    gate, gate_ckpt = train_crossfit_gate(
        name,
        horizon,
        oof_x,
        oof_abc,
        val_data[
            "feature"
        ],
        val_data[
            "abc"
        ],
    )

    scalar_alpha, scalar_curve = choose_scalar(
        val_data[
            "abc"
        ]
    )

    raw_val_alpha = gate_alpha(
        gate,
        gate_ckpt,
        val_data[
            "feature"
        ],
    )

    shrink_lambda, lambda_curve = choose_lambda(
        val_data[
            "abc"
        ],
        raw_val_alpha,
        scalar_alpha,
    )

    scalar_curve[
        "Dataset"
    ] = name

    scalar_curve[
        "Horizon"
    ] = horizon

    scalar_curve[
        "Kind"
    ] = "ScalarAlpha"

    lambda_curve[
        "Dataset"
    ] = name

    lambda_curve[
        "Horizon"
    ] = horizon

    lambda_curve[
        "Kind"
    ] = "ShrinkLambda"

    lambda_curve[
        "ScalarAlpha"
    ] = scalar_alpha

    calibration_frames.extend(
        [
            scalar_curve,
            lambda_curve,
        ]
    )

    if calibration_frames:
        pd.concat(
            calibration_frames,
            ignore_index=True,
        ).to_csv(
            CALIBRATION_PATH,
            index=False,
        )

    print(
        f"Validation calibration | "
        f"alpha0={scalar_alpha:.2f} | "
        f"lambda={shrink_lambda:.2f} | "
        f"gateEpoch={gate_ckpt['BestEpoch']}"
    )

    del (
        oof_x,
        oof_abc,
        oof_parts,
        raw_val_alpha,
        val_data,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 5. Static train+validation test retrieval memory.
    # --------------------------------------------------------
    test_memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "val_end"
        ],
        horizon,
    )

    test_memory_gpu = memory_gpu_cached(
        name,
        horizon,
        "test_trainval_memory",
        retriever,
        test_memory,
        C,
    )

    test = test_evaluate(
        data,
        horizon,
        direct_model,
        retriever,
        test_memory_gpu,
        gate,
        gate_ckpt,
        scalar_alpha,
        shrink_lambda,
    )

    metrics = test[
        "metrics"
    ]

    direct_mse, direct_mae = metrics[
        "Direct"
    ]

    retrieval_mse, retrieval_mae = metrics[
        "Retrieval"
    ]

    scalar_mse, scalar_mae = metrics[
        "Scalar"
    ]

    raw_mse, raw_mae = metrics[
        "RawAdaptive"
    ]

    shrink_mse, shrink_mae = metrics[
        "ShrinkAdaptive"
    ]

    oracle_mse, oracle_mae = metrics[
        "Oracle"
    ]

    # --------------------------------------------------------
    # 6. Paired moving-block bootstrap.
    # --------------------------------------------------------
    bootstrap_rows = [
        r
        for r in bootstrap_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    b_direct = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Direct"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            131313
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Direct-ShrinkAdaptive",
        **b_direct,
        "SignificantPositive":
            b_direct[
                "CI_Low"
            ]
            > 0.0,
    })

    b_scalar = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Scalar"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            232323
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Scalar-ShrinkAdaptive",
        **b_scalar,
        "SignificantPositive":
            b_scalar[
                "CI_Low"
            ]
            > 0.0,
    })

    pd.DataFrame(
        bootstrap_rows
    ).to_csv(
        BOOTSTRAP_PATH,
        index=False,
    )

    # --------------------------------------------------------
    # 7. Channel-level diagnostics.
    # --------------------------------------------------------
    channel_df = pd.DataFrame({
        "ChannelIndex":
            np.arange(
                C
            ),
        "ChannelName":
            data[
                "columns"
            ],
        "Direct_MSE":
            test[
                "channel_direct_mse"
            ],
        "ShrinkAdaptive_MSE":
            test[
                "channel_shrink_mse"
            ],
    })

    channel_df[
        "Improvement"
    ] = (
        channel_df[
            "Direct_MSE"
        ]
        - channel_df[
            "ShrinkAdaptive_MSE"
        ]
    )

    channel_df[
        "Improvement_pct"
    ] = (
        100.0
        * channel_df[
            "Improvement"
        ]
        / channel_df[
            "Direct_MSE"
        ]
    )

    channel_df.to_csv(
        DIRS[
            "channel"
        ]
        / (
            f"{name}_H{horizon}_"
            "channel_mse.csv"
        ),
        index=False,
    )

    improved_channel_fraction = float(
        (
            channel_df[
                "Improvement"
            ]
            > 0.0
        ).mean()
    )

    # --------------------------------------------------------
    # 8. Final summary.
    # --------------------------------------------------------
    exp18_mse = float(
        ref[
            "FullStride1_MSE"
        ]
    )

    exp18_mae = float(
        ref[
            "FullStride1_MAE"
        ]
    )

    row = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Channels":
            C,
        "OfficialPatchTST_MSE":
            direct_mse,
        "OfficialPatchTST_MAE":
            direct_mae,
        "Experiment18Reference_MSE":
            exp18_mse,
        "Experiment18Reference_MAE":
            exp18_mae,
        "DirectParityAbsDiff":
            abs(
                direct_mse
                - exp18_mse
            ),
        "Retrieval_MSE":
            retrieval_mse,
        "Retrieval_MAE":
            retrieval_mae,
        "ScalarAlpha":
            scalar_alpha,
        "Scalar_MSE":
            scalar_mse,
        "Scalar_MAE":
            scalar_mae,
        "RawAdaptive_MSE":
            raw_mse,
        "RawAdaptive_MAE":
            raw_mae,
        "ShrinkLambda":
            shrink_lambda,
        "ShrinkAdaptive_MSE":
            shrink_mse,
        "ShrinkAdaptive_MAE":
            shrink_mae,
        "Oracle_MSE":
            oracle_mse,
        "Oracle_MAE":
            oracle_mae,
        "RawMeanAlpha":
            test[
                "raw_mean_alpha"
            ],
        "ShrinkMeanAlpha":
            test[
                "shrink_mean_alpha"
            ],
        "OracleMeanAlpha":
            test[
                "oracle_mean_alpha"
            ],
        "OraclePositiveFraction":
            test[
                "oracle_positive_fraction"
            ],
        "ImprovedChannelFraction":
            improved_channel_fraction,
        "ScalarGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - scalar_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - shrink_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsScalar_pct":
            (
                100.0
                * (
                    scalar_mse
                    - shrink_mse
                )
                / scalar_mse
            ),
        "OracleHeadroomFromDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - oracle_mse
                )
                / direct_mse
            ),
        "OracleHeadroomFromShrink_pct":
            (
                100.0
                * (
                    shrink_mse
                    - oracle_mse
                )
                / shrink_mse
            ),
        "OfficialDirectBestEpoch":
            direct_epochs,
        "FrozenRetrieverBestEpoch":
            retriever_ckpt.get(
                "BestEpoch",
                np.nan,
            ),
        "GateBestEpoch":
            int(
                gate_ckpt[
                    "BestEpoch"
                ]
            ),
        "TestMemoryPerChannel":
            int(
                test_memory[
                    "M"
                ]
            ),
        "TestWindows":
            len(
                test[
                    "anchors"
                ]
            ),
        "RuntimeMinutes":
            (
                time.time()
                - start_time
            )
            / 60.0,
    }

    summary_rows = [
        r
        for r in summary_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    summary_rows.append(
        row
    )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    np.savez_compressed(
        DIRS[
            "paired"
        ]
        / (
            f"{name}_H{horizon}_"
            "anchor_mse.npz"
        ),
        TestAnchors=
            test[
                "anchors"
            ],
        Direct=
            test[
                "anchor_mse"
            ][
                "Direct"
            ],
        Retrieval=
            test[
                "anchor_mse"
            ][
                "Retrieval"
            ],
        Scalar=
            test[
                "anchor_mse"
            ][
                "Scalar"
            ],
        RawAdaptive=
            test[
                "anchor_mse"
            ][
                "RawAdaptive"
            ],
        ShrinkAdaptive=
            test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ],
        Oracle=
            test[
                "anchor_mse"
            ][
                "Oracle"
            ],
    )

    print(
        "\nFINAL CONDITION RESULT"
    )

    display(
        pd.DataFrame([
            row
        ])[
            [
                "Dataset",
                "Horizon",
                "OfficialPatchTST_MSE",
                "ShrinkAdaptive_MSE",
                "ShrinkGainVsDirect_pct",
                "Retrieval_MSE",
                "Scalar_MSE",
                "RawAdaptive_MSE",
                "Oracle_MSE",
                "ScalarAlpha",
                "ShrinkLambda",
                "ShrinkMeanAlpha",
                "ImprovedChannelFraction",
                "OracleHeadroomFromDirect_pct",
            ]
        ]
    )

    display(
        pd.DataFrame(
            bootstrap_rows
        )[
            (
                pd.DataFrame(
                    bootstrap_rows
                )[
                    "Dataset"
                ]
                == name
            )
            & (
                pd.DataFrame(
                    bootstrap_rows
                )[
                    "Horizon"
                ]
                == horizon
            )
        ]
    )

    del (
        direct_model,
        direct_ckpt,
        retriever,
        retriever_ckpt,
        gate,
        gate_ckpt,
        test_memory,
        test_memory_gpu,
        test,
        channel_df,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_df
)



######################################################################################################################################################
Weather | H=96 | OFFICIAL PATCHTST + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 18 reference: MSE=0.149690, MAE=0.197816, best_epoch=65
Frozen retriever best epoch: 8
Fold direct Weather     H= 96 F1 ep=001/65 train=0.951592 lr=1.000e-04
Fold direct Weather     H= 96 F1 ep=002/65 train=0.609418 lr=1.000e-04
Fold direct Weather     H= 96 F1 ep=003/65 train=0.529759 lr=1.000e-04
Fold direct Weather     H= 96 F1 ep=004/65 train=0.521188 lr=9.000e-05
Fold direct Weather     H= 96 F1 ep=005/65 train=0.514889 lr=8.100e-05
Fold direct Weather     H= 96 F1 ep=006/65 train=0.508319 lr=7.290e-05
Fold direct Weather     H= 96 F1 ep=007/65 train=0.501999 lr=6.561e-05
Fold direct Weather     H= 96 F1 ep=008/

,Dataset,Horizon,OfficialPatchTST_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct
0,Weather,96,0.14969,0.14744,1.503275,0.240003,0.14836,0.14744,0.123863,0.1,1.0,0.159055,0.380952,17.254005


,Dataset,Horizon,Comparison,MeanImprovement,CI_Low,CI_High,SignificantPositive
0,Weather,96,Direct-ShrinkAdaptive,0.00225,0.001320,0.003172,True
1,Weather,96,Scalar-ShrinkAdaptive,0.00092,0.000163,0.001656,True



######################################################################################################################################################
Weather | H=192 | OFFICIAL PATCHTST + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 18 reference: MSE=0.194775, MAE=0.240853, best_epoch=45
Frozen retriever best epoch: 5
Fold direct Weather     H=192 F1 ep=001/45 train=0.981854 lr=1.000e-04
Fold direct Weather     H=192 F1 ep=002/45 train=0.686572 lr=1.000e-04
Fold direct Weather     H=192 F1 ep=003/45 train=0.624206 lr=1.000e-04
Fold direct Weather     H=192 F1 ep=004/45 train=0.616110 lr=9.000e-05
Fold direct Weather     H=192 F1 ep=005/45 train=0.608612 lr=8.100e-05
Fold direct Weather     H=192 F1 ep=006/45 train=0.603664 lr=7.290e-05
Fold direct Weather     H=192 F1 ep=007/45 train=0.600255 lr=6.561e-05
Fold direct Weather     H=192 F1 ep=008

,Dataset,Horizon,OfficialPatchTST_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct
0,Weather,192,0.194775,0.190482,2.203907,0.300271,0.192337,0.190482,0.161471,0.1,1.0,0.110582,0.666667,17.09888


...

,Dataset,Horizon,Comparison,MeanImprovement,CI_Low,CI_High,SignificantPositive
6,Weather,720,Direct-ShrinkAdaptive,0.006829,0.005369,0.008341,True
7,Weather,720,Scalar-ShrinkAdaptive,0.001146,0.000210,0.002089,True



######################################################################################################################################################
Electricity | H=96 | OFFICIAL PATCHTST + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 18 reference: MSE=0.129977, MAE=0.222527, best_epoch=97
Frozen retriever best epoch: 7
Fold direct Electricity H= 96 F1 ep=001/97 train=0.500946 lr=4.591e-06
Fold direct Electricity H= 96 F1 ep=002/97 train=0.287236 lr=6.350e-06
Fold direct Electricity H= 96 F1 ep=003/97 train=0.239492 lr=9.233e-06
Fold direct Electricity H= 96 F1 ep=004/97 train=0.207848 lr=1.317e-05
Fold direct Electricity H= 96 F1 ep=005/97 train=0.188400 lr=1.806e-05
Fold direct Electricity H= 96 F1 ep=006/97 train=0.177077 lr=2.379e-05
Fold direct Electricity H= 96 F1 ep=007/97 train=0.170641 lr=3.022e-05
Fold direct Electricity H= 96 F1 ep=

,Dataset,Horizon,OfficialPatchTST_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct
0,Electricity,96,0.129977,0.129218,0.584101,1.839673,0.129977,0.129218,0.119747,0.0,1.0,0.020581,0.417445,7.870349


,Dataset,Horizon,Comparison,MeanImprovement,CI_Low,CI_High,SignificantPositive
8,Electricity,96,Direct-ShrinkAdaptive,0.000759,0.000457,0.001120,True
9,Electricity,96,Scalar-ShrinkAdaptive,0.000759,0.000463,0.001113,True



######################################################################################################################################################
Electricity | H=192 | OFFICIAL PATCHTST + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 18 reference: MSE=0.149116, MAE=0.242038, best_epoch=48
Frozen retriever best epoch: 16
Fold direct Electricity H=192 F1 ep=001/48 train=0.533017 lr=4.591e-06
Fold direct Electricity H=192 F1 ep=002/48 train=0.298756 lr=6.350e-06
Fold direct Electricity H=192 F1 ep=003/48 train=0.251623 lr=9.233e-06
Fold direct Electricity H=192 F1 ep=004/48 train=0.220620 lr=1.317e-05
Fold direct Electricity H=192 F1 ep=005/48 train=0.201749 lr=1.806e-05
Fold direct Electricity H=192 F1 ep=006/48 train=0.191237 lr=2.379e-05
Fold direct Electricity H=192 F1 ep=007/48 train=0.184810 lr=3.022e-05
Fold direct Electricity H=192 F1 e


## 16. Compact decision table

가장 먼저 볼 값은 다음입니다.

$$
\text{Gain}
=
100
\cdot
\frac{
\mathrm{MSE}_{\mathrm{Direct}}
-
\mathrm{MSE}_{\mathrm{Ours}}
}{
\mathrm{MSE}_{\mathrm{Direct}}
}
$$

`ShrinkGainVsDirect_pct > 0`이면 Ours가 strong PatchTST보다 좋습니다.

통계적 안정성은 moving-block bootstrap에서 확인합니다.

$$
\mathrm{CI}_{\mathrm{low}} > 0
$$

이면 Direct 대비 improvement가 유의한 것으로 해석합니다.


In [ ]:

if not len(
    summary_df
):
    raise RuntimeError(
        "No completed Experiment 19 result."
    )

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "OfficialPatchTST_MSE",
        "ShrinkAdaptive_MSE",
        "ShrinkGainVsDirect_pct",
        "OfficialPatchTST_MAE",
        "ShrinkAdaptive_MAE",
        "ScalarAlpha",
        "ShrinkLambda",
        "ShrinkMeanAlpha",
        "ImprovedChannelFraction",
        "Oracle_MSE",
        "OracleHeadroomFromDirect_pct",
        "DirectParityAbsDiff",
    ]
].copy()

compact[
    "Winner"
] = np.where(
    compact[
        "ShrinkAdaptive_MSE"
    ]
    < compact[
        "OfficialPatchTST_MSE"
    ],
    "Ours",
    "Direct",
)

display(
    compact
)

if BOOTSTRAP_PATH.exists():
    boot_df = pd.read_csv(
        BOOTSTRAP_PATH
    )

    direct_boot = boot_df[
        boot_df[
            "Comparison"
        ]
        == "Direct-ShrinkAdaptive"
    ][
        [
            "Dataset",
            "Horizon",
            "MeanImprovement",
            "CI_Low",
            "CI_High",
            "SignificantPositive",
        ]
    ].sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )

    display(
        direct_boot
    )



# 17. 해석 기준

## 시나리오 A — Weather와 Electricity에서 여러 horizon 개선

가장 좋은 결과입니다.

특히 strong official-level PatchTST를 대상으로 반복적인 개선이 나오면

$$
\boxed{
\text{historical memory is complementary to a strong parametric forecaster}
}
$$

라는 주장을 직접 지지합니다.

1% 안팎의 개선도 strong baseline 위에서 반복되면 충분히 의미가 있습니다.

---

## 시나리오 B — Weather만 개선

Weather는 이전 screening에서 historical retrieval이 전반적으로 유용했던 데이터셋입니다.

이 경우 연구 질문은

> historical memory는 어떤 data regime에서 strong forecaster를 보완하는가?

로 좁혀질 수 있습니다.

ETT와 Electricity 결과를 함께 사용해
retrieval-friendly regime의 특성을 분석할 수 있습니다.

---

## 시나리오 C — Electricity에서 scalar는 실패하지만 adaptive gate가 개선

이 결과도 매우 흥미롭습니다.

$$
\alpha_0 = 0
$$

인데 adaptive gate가 strong PatchTST를 개선하면

> retrieval is not globally useful, but is selectively useful for identifiable queries

라는 해석이 가능합니다.

---

## 시나리오 D — 둘 다 개선하지 못함

그 경우 현재 frozen method는 strong PatchTST에 대한 일반적인 model-agnostic augmentation이라고 주장할 수 없습니다.

하지만 Oracle headroom을 함께 봅니다.

### Oracle headroom도 작음

PatchTST가 historical analog의 정보를 대부분 이미 흡수하고 있을 가능성이 큽니다.

### Oracle headroom은 큼

보완 정보는 존재하지만 현재 retriever 또는 trust estimator가 이를 충분히 활용하지 못하는 것입니다.

---

## 중요한 원칙

Experiment 19의 Weather/Electricity test 결과를 본 뒤
retriever나 gate를 해당 데이터셋에 맞춰 다시 튜닝하지 않습니다.

먼저 8개 조건 전체 결과를 고정된 방법으로 끝까지 확인한 후 다음 연구 결정을 내립니다.


## 18. Saved artifacts

In [ ]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )


In [19]:
# ============================================================
# Experiment 19 — Disk-based progress / result checker
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os
from datetime import datetime

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_plus_frozen_retrieval"
)

EXP18_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_baselines"
)

TASKS = [
    ("Weather", 96),
    ("Weather", 192),
    ("Weather", 336),
    ("Weather", 720),
    ("Electricity", 96),
    ("Electricity", 192),
    ("Electricity", 336),
    ("Electricity", 720),
]

summary_path = ROOT / "summary.csv"
bootstrap_path = ROOT / "bootstrap.csv"
folds_path = ROOT / "folds.csv"

exp18_summary = pd.read_csv(EXP18_ROOT / "summary.csv")

if summary_path.exists():
    summary = pd.read_csv(summary_path)
else:
    summary = pd.DataFrame()

rows = []

for dataset, h in TASKS:

    # Experiment 18에서 정한 fold 학습 epoch
    ref = exp18_summary[
        (exp18_summary["Dataset"] == dataset)
        & (exp18_summary["Horizon"] == h)
    ]

    target_epoch = (
        int(ref.iloc[0]["BestEpoch"])
        if len(ref) == 1
        else None
    )

    # 최종 결과 존재 여부
    completed = (
        len(summary) > 0
        and (
            (summary["Dataset"] == dataset)
            & (summary["Horizon"] == h)
        ).any()
    )

    # Fold별 상태
    fold_status = []

    for fold in [1, 2, 3]:
        ckpt = (
            ROOT / "fold_direct"
            / f"{dataset}_H{h}_F{fold}_official_direct.pt"
        )

        oof = (
            ROOT / "oof"
            / f"{dataset}_H{h}_F{fold}_official_direct.npz"
        )

        hist = (
            ROOT / "history"
            / f"{dataset}_H{h}_F{fold}_direct_history.csv"
        )

        last_epoch = 0

        if hist.exists():
            try:
                hh = pd.read_csv(hist)
                if len(hh):
                    last_epoch = int(hh["Epoch"].max())
            except Exception:
                pass

        if oof.exists():
            s = "OOF done"
        elif ckpt.exists():
            s = "Direct done"
        elif last_epoch > 0:
            s = f"epoch {last_epoch}/{target_epoch}"
        else:
            s = "-"

        fold_status.append(s)

    val_cache = (
        ROOT / "validation"
        / f"{dataset}_H{h}_validation_features.npz"
    ).exists()

    gate = (
        ROOT / "gate"
        / f"{dataset}_H{h}_official_direct.pt"
    ).exists()

    paired = (
        ROOT / "paired_test"
        / f"{dataset}_H{h}_anchor_mse.npz"
    ).exists()

    channel = (
        ROOT / "channel_test"
        / f"{dataset}_H{h}_channel_mse.csv"
    ).exists()

    if completed:
        status = "✅ COMPLETE"
    elif paired:
        status = "⚠️ Test done / summary missing"
    elif gate:
        status = "🟡 Gate done"
    elif val_cache:
        status = "🟡 Validation done"
    elif any(x != "-" for x in fold_status):
        status = "🟠 Cross-fit in progress"
    else:
        status = "⚪ Not started"

    rows.append({
        "Dataset": dataset,
        "H": h,
        "Status": status,
        "TargetEpoch": target_epoch,
        "Fold1": fold_status[0],
        "Fold2": fold_status[1],
        "Fold3": fold_status[2],
        "ValCache": val_cache,
        "Gate": gate,
        "PairedTest": paired,
        "ChannelResult": channel,
    })

status_df = pd.DataFrame(rows)

print("=" * 120)
print("EXPERIMENT 19 DISK STATUS")
print("=" * 120)
display(status_df)

print("\n" + "=" * 120)
print("SAVED FINAL RESULTS")
print("=" * 120)

if len(summary):
    cols = [
        "Dataset",
        "Horizon",
        "OfficialPatchTST_MSE",
        "ShrinkAdaptive_MSE",
        "ShrinkGainVsDirect_pct",
        "ScalarAlpha",
        "ShrinkLambda",
        "Oracle_MSE",
        "OracleHeadroomFromDirect_pct",
    ]

    cols = [c for c in cols if c in summary.columns]

    display(
        summary[cols]
        .sort_values(["Dataset", "Horizon"])
        .reset_index(drop=True)
    )

    print(f"\nCompleted conditions: {len(summary)}/8")
else:
    print("summary.csv가 아직 없습니다.")

print("\n" + "=" * 120)
print("LATEST MODIFIED FILES")
print("=" * 120)

files = [
    p for p in ROOT.rglob("*")
    if p.is_file()
]

files = sorted(
    files,
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)[:20]

for p in files:
    t = datetime.fromtimestamp(
        p.stat().st_mtime
    ).strftime("%Y-%m-%d %H:%M:%S")

    print(
        t,
        " | ",
        p.relative_to(ROOT)
    )

print("\n" + "=" * 120)

if len(summary) == 8:
    print("✅ Experiment 19 전체 8개 조건이 완료되었습니다.")
else:
    print(
        f"현재 최종 완료: {len(summary)}/8 조건"
    )
    print(
        "미완료 상태라면 RESUME=True 상태에서 "
        "Main Experiment 셀을 다시 실행하면 됩니다."
    )

EXPERIMENT 19 DISK STATUS


,Dataset,H,Status,TargetEpoch,Fold1,Fold2,Fold3,ValCache,Gate,PairedTest,ChannelResult
0,Weather,96,✅ COMPLETE,65,OOF done,OOF done,OOF done,True,True,True,True
1,Weather,192,✅ COMPLETE,45,OOF done,OOF done,OOF done,True,True,True,True
2,Weather,336,✅ COMPLETE,19,OOF done,OOF done,OOF done,True,True,True,True
3,Weather,720,✅ COMPLETE,23,OOF done,OOF done,OOF done,True,True,True,True
4,Electricity,96,✅ COMPLETE,97,OOF done,OOF done,OOF done,True,True,True,True
5,Electricity,192,✅ COMPLETE,48,OOF done,OOF done,OOF done,True,True,True,True
6,Electricity,336,✅ COMPLETE,69,OOF done,OOF done,OOF done,True,True,True,True
7,Electricity,720,✅ COMPLETE,39,OOF done,OOF done,OOF done,True,True,True,True



SAVED FINAL RESULTS


,Dataset,Horizon,OfficialPatchTST_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,ScalarAlpha,ShrinkLambda,Oracle_MSE,OracleHeadroomFromDirect_pct
0,Electricity,96,0.129977,0.129218,0.584101,0.0,1.00,0.119747,7.870349
1,Electricity,192,0.149116,0.148559,0.373992,0.0,0.75,0.139304,6.580320
2,Electricity,336,0.165690,0.164748,0.568902,0.0,1.00,0.154490,6.760021
3,Electricity,720,0.203216,0.202541,0.332348,0.0,1.00,0.187749,7.611145
4,Weather,96,0.149690,0.147440,1.503275,0.1,1.00,0.123863,17.254005
5,Weather,192,0.194775,0.190482,2.203907,0.1,1.00,0.161471,17.098880
6,Weather,336,0.246900,0.241303,2.267156,0.2,0.75,0.204120,17.327073
7,Weather,720,0.320894,0.314066,2.128003,0.1,1.00,0.265141,17.374355



Completed conditions: 8/8

LATEST MODIFIED FILES
2026-08-26 22:13:59  |  paired_test/Electricity_H720_anchor_mse.npz
2026-08-26 22:13:59  |  summary.csv
2026-08-26 22:13:59  |  bootstrap.csv
2026-08-26 22:13:59  |  channel_test/Electricity_H720_channel_mse.csv
2026-08-26 22:13:27  |  memory_embeddings/Electricity_H720_test_trainval_memory_emb.npy
2026-08-26 22:13:24  |  calibration.csv
2026-08-26 22:13:24  |  history/Electricity_H720_gate_history.csv
2026-08-26 22:13:24  |  gate/Electricity_H720_official_direct.pt
2026-08-26 22:13:04  |  validation/Electricity_H720_validation_features.npz
2026-08-26 22:12:50  |  memory_embeddings/Electricity_H720_validation_train_memory_emb.npy
2026-08-26 22:12:47  |  folds.csv
2026-08-26 22:12:47  |  oof/Electricity_H720_F3_official_direct.npz
2026-08-26 22:12:43  |  memory_embeddings/Electricity_H720_F3_prefix15650_emb.npy
2026-08-26 22:12:41  |  fold_direct/Electricity_H720_F3_official_direct.pt
2026-08-26 22:12:41  |  history/Electricity_H720_F3_d